# Feature Engineering and Ridge Baseline

This notebook starts from the reproducible interim datasets, defines the chronological evaluation design, and retains only the feature choices supported by walk-forward validation. The 2024/25 test season remains untouched.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## Load prepared data

The expensive raw-table joins live in `src/prem_valuation/data.py`. This notebook loads only the resulting player-season tables.

In [ ]:
model_data = pd.read_csv(
    "../data/interim/labelled_player_seasons.csv.gz",
    parse_dates=["season_end_date", "valuation_date", "date_of_birth"],
)

scoring_data = pd.read_csv(
    "../data/interim/scoring_2025_26.csv.gz",
    parse_dates=["season_end_date", "date_of_birth"],
)

In [ ]:
def add_engineered_features(data):
    result = data.copy()
    result["age_distance_squared"] = (
        result["age_at_season_end"] - 25
    ) ** 2

    stable_minutes = result["minutes_played"].clip(lower=450)
    result["goals_per_90_stable"] = (
        result["goals"] / stable_minutes * 90
    )
    result["assists_per_90_stable"] = (
        result["assists"] / stable_minutes * 90
    )
    return result

model_data = add_engineered_features(model_data)
scoring_data = add_engineered_features(scoring_data)

## Chronological split

- Train: 2016/17–2022/23
- Validation: 2023/24
- Final untouched test: 2024/25
- Unlabelled scoring population: 2025/26

In [ ]:
train_data = model_data.loc[model_data["season"].between(2016, 2022)].copy()
validation_data = model_data.loc[model_data["season"].eq(2023)].copy()
test_data = model_data.loc[model_data["season"].eq(2024)].copy()

for name, dataset in {
    "train": train_data,
    "validation": validation_data,
    "test": test_data,
    "scoring": scoring_data,
}.items():
    print(name, dataset.shape, sorted(dataset["season"].unique()))

## Feature specification

Identifiers, names, dates, and valuation timing metadata are excluded. Citizenship is excluded because it is high-cardinality and reflects a current rather than historical profile. `season` is retained only for splitting, not as a feature.

In [ ]:
target = "market_value_in_eur"

original_numeric_features = [
    "appearances", "minutes_played", "goals", "assists",
    "yellow_cards", "red_cards", "clubs_played_for",
    "height_in_cm", "age_at_season_end",
]
original_categorical_features = ["position", "foot"]

selected_numeric_features = original_numeric_features + [
    "age_distance_squared"
]
selected_categorical_features = ["position", "sub_position", "foot"]

rate_numeric_features = selected_numeric_features + [
    "goals_per_90_stable", "assists_per_90_stable"
]
selected_features = selected_numeric_features + selected_categorical_features

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

## Reusable temporal evaluation

Each fold trains only on earlier seasons and validates on the next season. MAE is the primary metric; RMSE and R² remain diagnostic.

In [ ]:
def make_ridge_model(numerical_columns, categorical_columns, alpha=1.0):
    preprocessor = ColumnTransformer(transformers=[
        ("numeric", numeric_transformer, numerical_columns),
        ("categorical", categorical_transformer, categorical_columns),
    ])
    return Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("regressor", Ridge(alpha=alpha)),
    ])


def evaluate_predictions(actual, predicted):
    return {
        "mae": mean_absolute_error(actual, predicted),
        "rmse": root_mean_squared_error(actual, predicted),
        "r2": r2_score(actual, predicted),
    }


def walk_forward_evaluate(numerical_columns, categorical_columns, alpha=1.0):
    rows = []
    feature_columns = numerical_columns + categorical_columns

    for validation_season in range(2019, 2024):
        fold_train = model_data.loc[
            model_data["season"].between(2016, validation_season - 1)
        ]
        fold_validation = model_data.loc[
            model_data["season"].eq(validation_season)
        ]
        model = make_ridge_model(
            numerical_columns, categorical_columns, alpha=alpha
        )
        model.fit(fold_train[feature_columns], fold_train[target])
        predictions = np.clip(
            model.predict(fold_validation[feature_columns]), 0, None
        )
        rows.append({
            "validation_season": validation_season,
            **evaluate_predictions(fold_validation[target], predictions),
        })

    return pd.DataFrame(rows)

## Ridge model selection

In [ ]:
original_cv = walk_forward_evaluate(
    original_numeric_features, original_categorical_features
)
age_cv = walk_forward_evaluate(
    selected_numeric_features, original_categorical_features
)
selected_cv = walk_forward_evaluate(
    selected_numeric_features, selected_categorical_features
)
rates_cv = walk_forward_evaluate(
    rate_numeric_features, selected_categorical_features
)

cv_summary = pd.DataFrame({
    "original": original_cv[["mae", "rmse", "r2"]].mean(),
    "age_curve": age_cv[["mae", "rmse", "r2"]].mean(),
    "age_and_subposition": selected_cv[["mae", "rmse", "r2"]].mean(),
    "plus_stable_rates": rates_cv[["mae", "rmse", "r2"]].mean(),
}).T

cv_summary

In [ ]:
alpha_values = [0.01, 0.1, 1, 10, 100, 1000]

alpha_results = []

for alpha in alpha_values:
    fold_results = walk_forward_evaluate(
        selected_numeric_features,
        selected_categorical_features,
        alpha=alpha,
    )

    alpha_results.append({
        "alpha": alpha,
        "mae": fold_results["mae"].mean(),
        "rmse": fold_results["rmse"].mean(),
        "r2": fold_results["r2"].mean(),
    })

alpha_summary = (
    pd.DataFrame(alpha_results)
    .sort_values("mae")
    .reset_index(drop=True)
)

alpha_summary

In [ ]:
alpha_1_folds = walk_forward_evaluate(
    selected_numeric_features,
    selected_categorical_features,
    alpha=1,
)

alpha_100_folds = walk_forward_evaluate(
    selected_numeric_features,
    selected_categorical_features,
    alpha=100,
)

alpha_fold_comparison = alpha_1_folds[
    ["validation_season", "mae", "rmse", "r2"]
].merge(
    alpha_100_folds[["validation_season", "mae", "rmse", "r2"]],
    on="validation_season",
    suffixes=("_alpha_1", "_alpha_100"),
)

alpha_fold_comparison["mae_improvement"] = (
    alpha_fold_comparison["mae_alpha_1"]
    - alpha_fold_comparison["mae_alpha_100"]
)

alpha_fold_comparison

## Final Ridge evaluation

In [ ]:
selected_alpha = 100
development_data = pd.concat(
    [train_data, validation_data],
    ignore_index=True,
)

final_model = make_ridge_model(
    selected_numeric_features,
    selected_categorical_features,
    alpha=selected_alpha,
)

final_model.fit(
    development_data[selected_features],
    development_data[target],
)

test_predictions = np.clip(
    final_model.predict(test_data[selected_features]),
    0,
    None,
)

pd.Series(
    evaluate_predictions(test_data[target], test_predictions),
    name="2024/25 final test",
)

## Test residual analysis

In [ ]:
test_results = test_data[
    [
        "season",
        "player_id",
        "player_name",
        "position",
        "sub_position",
        "age_at_season_end",
        "minutes_played",
        "market_value_in_eur",
    ]
].copy()

test_results["predicted_value"] = test_predictions

test_results["residual"] = (
    test_results["market_value_in_eur"]
    - test_results["predicted_value"]
)

test_results["absolute_error"] = test_results["residual"].abs()

test_results.head()

In [ ]:
largest_errors = test_results.nlargest(
    15,
    "absolute_error",
)[
    [
        "player_name",
        "position",
        "age_at_season_end",
        "minutes_played",
        "market_value_in_eur",
        "predicted_value",
        "residual",
        "absolute_error",
    ]
]

largest_errors

In [ ]:
test_results["valuation_gap"] = (
    test_results["predicted_value"]
    - test_results["market_value_in_eur"]
)

In [ ]:
position_analysis = test_results.groupby("position").agg(
    players=("player_id", "size"),
    actual_mean=("market_value_in_eur", "mean"),
    predicted_mean=("predicted_value", "mean"),
    mae=("absolute_error", "mean"),
    mean_residual=("residual", "mean"),
).sort_values("mae", ascending=False)

position_analysis

In [ ]:
test_results["value_band"] = pd.cut(
    test_results["market_value_in_eur"],
    bins=[0, 5_000_000, 15_000_000, 30_000_000, 60_000_000, np.inf],
    labels=["€0–5m", "€5–15m", "€15–30m", "€30–60m", "€60m+"],
    include_lowest=True,
)

value_band_analysis = test_results.groupby(
    "value_band",
    observed=False,
).agg(
    players=("player_id", "size"),
    actual_mean=("market_value_in_eur", "mean"),
    predicted_mean=("predicted_value", "mean"),
    mae=("absolute_error", "mean"),
    mean_residual=("residual", "mean"),
)

value_band_analysis

In [ ]:
test_results["minutes_band"] = pd.cut(
    test_results["minutes_played"],
    bins=[-1, 449, 899, 1799, np.inf],
    labels=["Under 450", "450–899", "900–1799", "1800+"],
)

minutes_analysis = test_results.groupby(
    "minutes_band",
    observed=False,
).agg(
    players=("player_id", "size"),
    actual_mean=("market_value_in_eur", "mean"),
    predicted_mean=("predicted_value", "mean"),
    mae=("absolute_error", "mean"),
    mean_residual=("residual", "mean"),
)

minutes_analysis

In [ ]:
test_results["age_band"] = pd.cut(
    test_results["age_at_season_end"],
    bins=[16, 20, 23, 26, 29, 32, 36, 50],
    right=False,
)

age_analysis = test_results.groupby(
    "age_band",
    observed=False,
).agg(
    players=("player_id", "size"),
    actual_mean=("market_value_in_eur", "mean"),
    predicted_mean=("predicted_value", "mean"),
    mae=("absolute_error", "mean"),
    mean_residual=("residual", "mean"),
)

age_analysis

## Residual analysis findings

The frozen Ridge baseline achieved a 2024/25 test MAE of €10.39m and R² of 0.426.

- Prediction errors were substantially larger for attackers and midfielders than goalkeepers.
- The model regressed valuations toward the middle: it overpredicted low-value players and underpredicted every player valued above €60m.
- Prime-age players aged 23–28 were systematically underpredicted, while players aged 32–35 were slightly overpredicted.
- Higher-minute players had larger absolute errors mainly because they were more valuable, not because additional minutes made predictions less reliable.
- Low-minute players should still be separated in final rankings because their performance statistics are based on weaker samples.
- The largest errors involved elite reputation, injuries and other information absent from the feature set.

These patterns suggest that market value contains nonlinear effects and a star premium that Ridge cannot represent adequately.

## Experiment log

| Experiment | 2023/24 result | Decision | Reason |
|---|---:|---|---|
| Median dummy | MAE €15.45m | Benchmark | Minimum useful baseline. |
| Log-target Ridge | MAE €11.69m | Rejected | Worse metrics and implausible €288m Palmer prediction. |
| Raw-target Ridge | MAE €10.89m | Retained foundation | Beat dummy and log target. |
| Numeric season trend | MAE €11.44m | Rejected | Improved RMSE/R² but worsened primary MAE. |
| Squared distance from age 25 | MAE €10.69m | Retained | Captures the observed nonlinear age-value curve. |
| Sub-position | MAE €10.38m | Retained | Improved all metrics and remained useful across temporal folds. |
| Stabilised goals/assists per 90 | MAE €10.35m | Rejected | Only marginal validation gain; mixed results over five folds. |
| Ridge alpha tuning | CV MAE €9.29m at alpha=100 | Retained | Best mean MAE across five folds and improved 4/5 seasons, though the gain over alpha=1 was marginal. |
| Frozen Ridge test | 2024/25 MAE €10.39m, R² 0.426 | Final baseline | Evaluated the fixed feature set and alpha after retraining through 2023/24. |

Walk-forward averages improved from MAE €9.63m / R² 0.477 for the original features to MAE €9.29m after feature selection and Ridge regularisation tuning.